<a href="https://colab.research.google.com/github/omsoom/ED-SSM/blob/main/Ed_ssm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass
import math

@dataclass
class ModelConfig:
    d_model: int = 64
    vocab_size: int = 256
    max_archive_size: int = 500
    vigilance_threshold: float = 0.85
    lr: float = 1e-3

class DualSlotNeurons(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.d_model = config.d_model
        # 256 bytes + 2 special tokens (SOS=256, EOS=257)
        self.byte_emb = nn.Embedding(config.vocab_size + 2, self.d_model)

        # Transformations for the two memory slots
        self.W_in = nn.Linear(self.d_model, self.d_model)
        self.W_out = nn.Linear(self.d_model, self.d_model)

        self.norm_in = nn.LayerNorm(self.d_model)
        self.norm_out = nn.LayerNorm(self.d_model)

    def initialize_memory(self, B, device):
        # Memory shape: (B, 256, 2, d_model) -> slot 0=Before, slot 1=After
        memory = torch.zeros(B, 256, 2, self.d_model, device=device)
        sos_emb = self.byte_emb(torch.tensor([256], device=device))
        eos_emb = self.byte_emb(torch.tensor([257], device=device))

        for i in range(256):
            self_emb = self.byte_emb(torch.tensor([i], device=device))
            memory[0, i, 0] = self.norm_in(self.W_in(sos_emb))  # Before = SOS
            memory[0, i, 1] = self.norm_out(self.W_out(eos_emb)) # After = EOS
        return memory

    def forward(self, x_curr, prev_byte, memory_neurons, verbose=False):
        B = x_curr.shape[0]
        device = x_curr.device
        batch_idx = torch.arange(B, device=device)

        new_memory = memory_neurons.clone()
        curr_emb = self.byte_emb(x_curr)

        # 1. Delayed Consequence Update & Flush
        if prev_byte is not None:
            prev_emb = self.byte_emb(prev_byte)

            # Update 'After' slot of previous node with current byte
            h_out_prev = self.W_out(curr_emb)
            new_memory[batch_idx, prev_byte, 1] = self.norm_out(h_out_prev)

            # Update 'Before' slot of current node with previous byte
            h_in_curr = self.W_in(prev_emb)
            new_memory[batch_idx, x_curr, 0] = self.norm_in(h_in_curr)

            if verbose: print(f"    [Update] Neuron {prev_byte.item()} After={x_curr.item()} | Neuron {x_curr.item()} Before={prev_byte.item()}")

            # Flush: concatenate Before slot, Self embedding, After slot
            h_in_prev = new_memory[batch_idx, prev_byte, 0]
            h_out_prev_normed = new_memory[batch_idx, prev_byte, 1]

            flush_msg = torch.cat([h_in_prev, prev_emb, h_out_prev_normed], dim=-1)
        else:
            # SOS flush
            sos_emb = self.byte_emb(torch.tensor([256], device=device))
            eos_emb = self.byte_emb(torch.tensor([257], device=device))
            # Update current node's Before with SOS
            new_memory[batch_idx, x_curr, 0] = self.norm_in(self.W_in(sos_emb))
            flush_msg = torch.cat([self.W_in(sos_emb), curr_emb, self.W_out(eos_emb)], dim=-1)

        return flush_msg, new_memory

class AttentionAggregator(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.d_model = config.d_model
        # Input to attention is d_model * 3 (Before + Self + After)
        self.W_q = nn.Linear(config.d_model * 3, config.d_model)
        self.W_k = nn.Linear(config.d_model * 3, config.d_model)
        self.W_v = nn.Linear(config.d_model * 3, config.d_model)
        self.layer_norm = nn.LayerNorm(config.d_model * 3)

    def forward(self, gated_flushes):
        normed_flushes = self.layer_norm(gated_flushes)
        Q = self.W_q(normed_flushes)
        K = self.W_k(normed_flushes)
        V = self.W_v(normed_flushes)
        attn_scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_model)
        attn_weights = F.softmax(attn_scores, dim=-1)
        path_output = torch.matmul(attn_weights, V)
        motif_vector = path_output.mean(dim=1)
        return motif_vector

class EDSSM_Final(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.neurons = DualSlotNeurons(config)
        self.aggregator = AttentionAggregator(config)
        self.lm_head = nn.Linear(config.d_model * 3, config.vocab_size)

    def forward(self, x_seq, memory_neurons, disable_gating=True, verbose=False):
        B, T = x_seq.shape
        device = x_seq.device

        prev_byte = None
        prev_logits = None
        gated_flushes = []
        logits_list = []

        for t in range(T):
            x_t = x_seq[:, t]
            if verbose: print(f"  Time Step {t}: Processing Byte {x_t.item()}")
            raw_flush, memory_neurons = self.neurons(x_t, prev_byte, memory_neurons, verbose=verbose)

            if prev_logits is not None:
                prob = F.softmax(prev_logits, dim=-1)[0, x_t.item()].item()
                surprise = -math.log(prob + 1e-8)
            else:
                surprise = 1.0

            gate = 1.0 if disable_gating else 1.0 + (self.config.alpha * surprise)
            gated_flush = raw_flush * gate
            gated_flushes.append(gated_flush)

            curr_logits = self.lm_head(raw_flush)
            logits_list.append(curr_logits)

            prev_logits = curr_logits
            prev_byte = x_t

        # FIX: Correctly reference self.neurons.byte_emb
        eos_emb = self.neurons.byte_emb(torch.tensor([257], device=device))
        last_emb = self.neurons.byte_emb(prev_byte)

        h_last_in = memory_neurons[torch.arange(B, device=device), prev_byte, 0]
        h_last_out = self.neurons.W_out(eos_emb)

        flush_input_eos = torch.cat([h_last_in, last_emb, h_last_out], dim=-1)
        gated_flushes.append(flush_input_eos * 1.0)

        flush_tensor = torch.stack(gated_flushes, dim=1)
        motif_vector = self.aggregator(flush_tensor)

        logits = torch.stack(logits_list, dim=1)
        return logits, motif_vector, memory_neurons

# ==========================================
# NEURON STATE DIAGNOSTIC
# ==========================================
def run_neuron_state_diagnostics():
    config = ModelConfig()
    model = EDSSM_Final(config)
    optimizer = torch.optim.Adam(model.parameters(), lr=config.lr)
    criterion = nn.CrossEntropyLoss()

    B = 1
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.to(device)

    memory_neurons = model.neurons.initialize_memory(B, device)

    print("Training on 'CAT' (50 steps) with Dual-Slot Neurons...")
    cat_seq = torch.tensor([[67, 65, 84]], dtype=torch.long).to(device)

    for step in range(50):
        optimizer.zero_grad()
        logits, _, memory_neurons = model(cat_seq, memory_neurons, disable_gating=True, verbose=False)
        memory_neurons = memory_neurons.detach()

        targets = cat_seq[:, 1:]
        loss = criterion(logits[:, :-1].contiguous().view(-1, config.vocab_size), targets.contiguous().view(-1))
        loss.backward()
        optimizer.step()

    print("Training Complete.\n")

    fresh_memory = model.neurons.initialize_memory(B, device)

    C = 67; A = 65; T = 84; R = 82; B_byte = 66

    print("="*50)
    print("PROBING INTERNAL DUAL-SLOT NEURON STATES")
    print("="*50)

    with torch.no_grad():
        print("\n--- Processing CAT ---")
        _, _, mem_cat = model(cat_seq, fresh_memory.clone(), disable_gating=True, verbose=True)
        h_A_cat = mem_cat[0, A] # Returns [h_in, h_out]

        print("\n--- Processing CAR ---")
        car_seq = torch.tensor([[C, A, R]], dtype=torch.long).to(device)
        _, _, mem_car = model(car_seq, fresh_memory.clone(), disable_gating=True, verbose=True)
        h_A_car = mem_car[0, A]

        print("\n--- Processing BAT ---")
        bat_seq = torch.tensor([[B_byte, A, T]], dtype=torch.long).to(device)
        _, _, mem_bat = model(bat_seq, fresh_memory.clone(), disable_gating=True, verbose=True)
        h_A_bat = mem_bat[0, A]

    def cos_sim(v1, v2):
        return F.cosine_similarity(v1.flatten().unsqueeze(0), v2.flatten().unsqueeze(0)).item()

    print("\n" + "="*50)
    print("FINAL NEURON STATE COMPARISONS (Full [Before, After] State)")
    print("="*50)
    print(f"sim(h_A in CAT, h_A in CAR): {cos_sim(h_A_cat, h_A_car):.4f}  (Divergence should be here: After T vs After R)")
    print(f"sim(h_A in CAT, h_A in BAT): {cos_sim(h_A_cat, h_A_bat):.4f}  (Divergence should be here: Before C vs Before B)")

if __name__ == "__main__":
    run_neuron_state_diagnostics()

Training on 'CAT' (50 steps) with Dual-Slot Neurons...
Training Complete.

PROBING INTERNAL DUAL-SLOT NEURON STATES

--- Processing CAT ---
  Time Step 0: Processing Byte 67
  Time Step 1: Processing Byte 65
    [Update] Neuron 67 After=65 | Neuron 65 Before=67
  Time Step 2: Processing Byte 84
    [Update] Neuron 65 After=84 | Neuron 84 Before=65

--- Processing CAR ---
  Time Step 0: Processing Byte 67
  Time Step 1: Processing Byte 65
    [Update] Neuron 67 After=65 | Neuron 65 Before=67
  Time Step 2: Processing Byte 82
    [Update] Neuron 65 After=82 | Neuron 82 Before=65

--- Processing BAT ---
  Time Step 0: Processing Byte 66
  Time Step 1: Processing Byte 65
    [Update] Neuron 66 After=65 | Neuron 65 Before=66
  Time Step 2: Processing Byte 84
    [Update] Neuron 65 After=84 | Neuron 84 Before=65

FINAL NEURON STATE COMPARISONS (Full [Before, After] State)
sim(h_A in CAT, h_A in CAR): 0.4397  (Divergence should be here: After T vs After R)
sim(h_A in CAT, h_A in BAT): 0.4437 

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass
import math

@dataclass
class ModelConfig:
    d_model: int = 64
    vocab_size: int = 256
    max_archive_size: int = 500
    vigilance_threshold: float = 0.85
    lr: float = 1e-3
    alpha: float = 1.0

class DualSlotNeurons(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.d_model = config.d_model
        self.byte_emb = nn.Embedding(config.vocab_size + 2, self.d_model)
        self.W_before = nn.Linear(self.d_model, self.d_model)
        self.W_after = nn.Linear(self.d_model, self.d_model)
        self.norm_before = nn.LayerNorm(self.d_model)
        self.norm_after = nn.LayerNorm(self.d_model)
        self.norm_flush = nn.LayerNorm(self.d_model * 3)

    def initialize_memory(self, B, device):
        memory = torch.zeros(B, 256, 2, self.d_model, device=device)
        sos_emb = self.byte_emb(torch.tensor([256], device=device))
        eos_emb = self.byte_emb(torch.tensor([257], device=device))
        for i in range(256):
            self_emb = self.byte_emb(torch.tensor([i], device=device))
            memory[0, i, 0] = self.norm_before(self.W_before(sos_emb))
            memory[0, i, 1] = self.norm_after(self.W_after(eos_emb))
        return memory

    def forward(self, x_curr, prev_byte, memory_neurons, verbose=False):
        B = x_curr.shape[0]
        device = x_curr.device
        batch_idx = torch.arange(B, device=device)

        new_memory = memory_neurons.clone()
        curr_emb = self.byte_emb(x_curr)

        if prev_byte is not None:
            prev_emb = self.byte_emb(prev_byte)
            new_memory[batch_idx, prev_byte, 1] = self.norm_after(self.W_after(curr_emb))
            new_memory[batch_idx, x_curr, 0] = self.norm_before(self.W_before(prev_emb))

            h_before_prev = new_memory[batch_idx, prev_byte, 0]
            h_after_prev = new_memory[batch_idx, prev_byte, 1]

            raw_flush = torch.cat([h_before_prev, prev_emb, h_after_prev], dim=-1)
            normed_flush = self.norm_flush(raw_flush)

        else:
            sos_emb = self.byte_emb(torch.tensor([256], device=device))
            eos_emb = self.byte_emb(torch.tensor([257], device=device))
            new_memory[batch_idx, x_curr, 0] = self.norm_before(self.W_before(sos_emb))
            raw_flush = torch.cat([self.W_before(sos_emb), curr_emb, self.W_after(eos_emb)], dim=-1)
            normed_flush = self.norm_flush(raw_flush)

        return normed_flush, new_memory

class MaxPoolAggregator(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.d_model = config.d_model
        # FIX: NO LAYERNORM HERE. We want the raw magnitudes.

    def forward(self, gated_flushes, verbose=False):
        if verbose:
            # Print the L2 norm of each flush to see if Entropy Gate is working
            norms = torch.norm(gated_flushes, dim=-1).squeeze(0)
            print(f"    [Aggregator Input Norms]: {[f'{n:.2f}' for n in norms.tolist()]}")

        # Max Pool over the sequence dimension
        motif_vector, max_indices = gated_flushes.max(dim=1)

        if verbose:
            # Which time step contributed the most features?
            unique_idxs, counts = torch.unique(max_indices, return_counts=True)
            dominant_idx = unique_idxs[torch.argmax(counts)]
            print(f"    [MaxPool]: Dominant flush index: {dominant_idx.item()}")

        return motif_vector

class EDSSM_Final(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.neurons = DualSlotNeurons(config)
        self.aggregator = MaxPoolAggregator(config)
        self.lm_head = nn.Linear(config.d_model * 3, config.vocab_size)

    def forward(self, x_seq, memory_neurons, disable_gating=False, verbose=False):
        B, T = x_seq.shape
        device = x_seq.device

        prev_byte = None
        prev_logits = None
        gated_flushes = []
        logits_list = []

        for t in range(T):
            x_t = x_seq[:, t]
            normed_flush, memory_neurons = self.neurons(x_t, prev_byte, memory_neurons, verbose=verbose)

            if prev_logits is not None:
                prob = F.softmax(prev_logits, dim=-1)[0, x_t.item()].item()
                surprise = -math.log(prob + 1e-8)
            else:
                surprise = 1.0

            gate = 1.0 if disable_gating else 1.0 + (self.config.alpha * surprise)

            gated_flush = normed_flush * gate
            gated_flushes.append(gated_flush)

            curr_logits = self.lm_head(normed_flush)
            logits_list.append(curr_logits)

            prev_logits = curr_logits
            prev_byte = x_t

        eos_emb = self.neurons.byte_emb(torch.tensor([257], device=device))
        last_emb = self.neurons.byte_emb(prev_byte)

        h_last_before = memory_neurons[torch.arange(B, device=device), prev_byte, 0]
        h_last_after = self.neurons.W_after(eos_emb)

        flush_input_eos = torch.cat([h_last_before, last_emb, h_last_after], dim=-1)
        normed_flush_eos = self.neurons.norm_flush(flush_input_eos)
        gated_flushes.append(normed_flush_eos * 1.0)

        flush_tensor = torch.stack(gated_flushes, dim=1)
        motif_vector = self.aggregator(flush_tensor, verbose=verbose)

        logits = torch.stack(logits_list, dim=1)
        return logits, motif_vector, memory_neurons

# ==========================================
# MOTIF & NEURON LEVEL DIAGNOSTIC
# ==========================================
def run_final_diagnostics():
    config = ModelConfig()
    model = EDSSM_Final(config)
    optimizer = torch.optim.Adam(model.parameters(), lr=config.lr)
    criterion = nn.CrossEntropyLoss()

    B = 1
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.to(device)

    memory_neurons = model.neurons.initialize_memory(B, device)

    print("Training on 'CAT' (50 steps) with Traced MaxPool Aggregator...")
    cat_seq = torch.tensor([[67, 65, 84]], dtype=torch.long).to(device)

    for step in range(50):
        optimizer.zero_grad()
        logits, _, memory_neurons = model(cat_seq, memory_neurons, disable_gating=False)
        memory_neurons = memory_neurons.detach()

        targets = cat_seq[:, 1:]
        loss = criterion(logits[:, :-1].contiguous().view(-1, config.vocab_size), targets.contiguous().view(-1))
        loss.backward()
        optimizer.step()

    print("Training Complete.\n")

    fresh_memory = model.neurons.initialize_memory(B, device)

    C = 67; A = 65; T = 84; R = 82; B_byte = 66

    print("="*50)
    print("FINAL ARCHITECTURE DIAGNOSTIC (Traced MaxPool)")
    print("="*50)

    with torch.no_grad():
        print("\n--- Processing CAT ---")
        _, motif_cat, mem_cat = model(cat_seq, fresh_memory.clone(), disable_gating=False, verbose=True)
        print("\n--- Processing CAR ---")
        car_seq = torch.tensor([[C, A, R]], dtype=torch.long).to(device)
        _, motif_car, mem_car = model(car_seq, fresh_memory.clone(), disable_gating=False, verbose=True)
        print("\n--- Processing BAT ---")
        bat_seq = torch.tensor([[B_byte, A, T]], dtype=torch.long).to(device)
        _, motif_bat, mem_bat = model(bat_seq, fresh_memory.clone(), disable_gating=False, verbose=True)

    def cos_sim(v1, v2):
        return F.cosine_similarity(v1.flatten().unsqueeze(0), v2.flatten().unsqueeze(0)).item()

    print("\n--- 1. NEURON-LEVEL SEPARATION (Internal Node 'A') ---")
    h_A_cat = mem_cat[0, A]
    h_A_car = mem_car[0, A]
    h_A_bat = mem_bat[0, A]
    print(f"sim(h_A in CAT, h_A in CAR): {cos_sim(h_A_cat, h_A_car):.4f}")
    print(f"sim(h_A in CAT, h_A in BAT): {cos_sim(h_A_cat, h_A_bat):.4f}")

    print("\n--- 2. MOTIF-LEVEL SEPARATION (Aggregator Output) ---")
    print(f"sim(Motif_CAT, Motif_CAR): {cos_sim(motif_cat, motif_car):.4f}")
    print(f"sim(Motif_CAT, Motif_BAT): {cos_sim(motif_cat, motif_bat):.4f}")
    print(f"sim(Motif_CAR, Motif_BAT): {cos_sim(motif_car, motif_bat):.4f}")

if __name__ == "__main__":
    run_final_diagnostics()

Training on 'CAT' (50 steps) with Traced MaxPool Aggregator...
Training Complete.

FINAL ARCHITECTURE DIAGNOSTIC (Traced MaxPool)

--- Processing CAT ---
    [Aggregator Input Norms]: ['28.68', '14.46', '14.28', '14.23']
    [MaxPool]: Dominant flush index: 0

--- Processing CAR ---
    [Aggregator Input Norms]: ['28.68', '14.46', '173.49', '14.18']
    [MaxPool]: Dominant flush index: 2

--- Processing BAT ---
    [Aggregator Input Norms]: ['28.62', '14.74', '14.80', '14.23']
    [MaxPool]: Dominant flush index: 0

--- 1. NEURON-LEVEL SEPARATION (Internal Node 'A') ---
sim(h_A in CAT, h_A in CAR): 0.5591
sim(h_A in CAT, h_A in BAT): 0.6160

--- 2. MOTIF-LEVEL SEPARATION (Aggregator Output) ---
sim(Motif_CAT, Motif_CAR): 0.6270
sim(Motif_CAT, Motif_BAT): 0.8897
sim(Motif_CAR, Motif_BAT): 0.5774


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass
import math

@dataclass
class ModelConfig:
    d_model: int = 64
    vocab_size: int = 256
    max_archive_size: int = 500
    vigilance_threshold: float = 0.85
    lr: float = 1e-3
    alpha: float = 1.0

class DualSlotNeurons(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.d_model = config.d_model
        self.byte_emb = nn.Embedding(config.vocab_size + 2, self.d_model)
        self.W_before = nn.Linear(self.d_model, self.d_model)
        self.W_after = nn.Linear(self.d_model, self.d_model)
        self.norm_before = nn.LayerNorm(self.d_model)
        self.norm_after = nn.LayerNorm(self.d_model)
        self.norm_flush = nn.LayerNorm(self.d_model * 3)

    def initialize_memory(self, B, device):
        memory = torch.zeros(B, 256, 2, self.d_model, device=device)
        sos_emb = self.byte_emb(torch.tensor([256], device=device))
        eos_emb = self.byte_emb(torch.tensor([257], device=device))
        for i in range(256):
            self_emb = self.byte_emb(torch.tensor([i], device=device))
            memory[0, i, 0] = self.norm_before(self.W_before(sos_emb))
            memory[0, i, 1] = self.norm_after(self.W_after(eos_emb))
        return memory

    def forward(self, x_curr, prev_byte, memory_neurons, verbose=False):
        B = x_curr.shape[0]
        device = x_curr.device
        batch_idx = torch.arange(B, device=device)

        new_memory = memory_neurons.clone()
        curr_emb = self.byte_emb(x_curr)

        if prev_byte is not None:
            prev_emb = self.byte_emb(prev_byte)
            new_memory[batch_idx, prev_byte, 1] = self.norm_after(self.W_after(curr_emb))
            new_memory[batch_idx, x_curr, 0] = self.norm_before(self.W_before(prev_emb))

            h_before_prev = new_memory[batch_idx, prev_byte, 0]
            h_after_prev = new_memory[batch_idx, prev_byte, 1]

            raw_flush = torch.cat([h_before_prev, prev_emb, h_after_prev], dim=-1)
            normed_flush = self.norm_flush(raw_flush)

        else:
            sos_emb = self.byte_emb(torch.tensor([256], device=device))
            eos_emb = self.byte_emb(torch.tensor([257], device=device))
            new_memory[batch_idx, x_curr, 0] = self.norm_before(self.W_before(sos_emb))
            raw_flush = torch.cat([self.W_before(sos_emb), curr_emb, self.W_after(eos_emb)], dim=-1)
            normed_flush = self.norm_flush(raw_flush)

        return normed_flush, new_memory

class SumAggregator(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.d_model = config.d_model

    def forward(self, gated_flushes, verbose=False):
        if verbose:
            norms = torch.norm(gated_flushes, dim=-1).squeeze(0)
            print(f"    [Aggregator Input Norms]: {[f'{n:.2f}' for n in norms.tolist()]}")

        # FIX: Simply sum the gated flushes.
        # The Entropy Gate ensures the suffix contributes more to the final direction.
        motif_vector = torch.sum(gated_flushes, dim=1)

        return motif_vector

class EDSSM_Final(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.neurons = DualSlotNeurons(config)
        self.aggregator = SumAggregator(config)
        self.lm_head = nn.Linear(config.d_model * 3, config.vocab_size)

    def forward(self, x_seq, memory_neurons, disable_gating=False, verbose=False):
        B, T = x_seq.shape
        device = x_seq.device

        prev_byte = None
        prev_logits = None
        gated_flushes = []
        logits_list = []

        for t in range(T):
            x_t = x_seq[:, t]
            normed_flush, memory_neurons = self.neurons(x_t, prev_byte, memory_neurons, verbose=verbose)

            if prev_logits is not None:
                prob = F.softmax(prev_logits, dim=-1)[0, x_t.item()].item()
                surprise = -math.log(prob + 1e-8)
            else:
                surprise = 1.0

            gate = 1.0 if disable_gating else 1.0 + (self.config.alpha * surprise)

            gated_flush = normed_flush * gate
            gated_flushes.append(gated_flush)

            curr_logits = self.lm_head(normed_flush)
            logits_list.append(curr_logits)

            prev_logits = curr_logits
            prev_byte = x_t

        eos_emb = self.neurons.byte_emb(torch.tensor([257], device=device))
        last_emb = self.neurons.byte_emb(prev_byte)

        h_last_before = memory_neurons[torch.arange(B, device=device), prev_byte, 0]
        h_last_after = self.neurons.W_after(eos_emb)

        flush_input_eos = torch.cat([h_last_before, last_emb, h_last_after], dim=-1)
        normed_flush_eos = self.neurons.norm_flush(flush_input_eos)
        gated_flushes.append(normed_flush_eos * 1.0)

        flush_tensor = torch.stack(gated_flushes, dim=1)
        motif_vector = self.aggregator(flush_tensor, verbose=verbose)

        logits = torch.stack(logits_list, dim=1)
        return logits, motif_vector, memory_neurons

# ==========================================
# MOTIF & NEURON LEVEL DIAGNOSTIC
# ==========================================
def run_final_diagnostics():
    config = ModelConfig()
    model = EDSSM_Final(config)
    optimizer = torch.optim.Adam(model.parameters(), lr=config.lr)
    criterion = nn.CrossEntropyLoss()

    B = 1
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.to(device)

    memory_neurons = model.neurons.initialize_memory(B, device)

    print("Training on 'CAT' (50 steps) with Sum Aggregator...")
    cat_seq = torch.tensor([[67, 65, 84]], dtype=torch.long).to(device)

    for step in range(50):
        optimizer.zero_grad()
        logits, _, memory_neurons = model(cat_seq, memory_neurons, disable_gating=False)
        memory_neurons = memory_neurons.detach()

        targets = cat_seq[:, 1:]
        loss = criterion(logits[:, :-1].contiguous().view(-1, config.vocab_size), targets.contiguous().view(-1))
        loss.backward()
        optimizer.step()

    print("Training Complete.\n")

    fresh_memory = model.neurons.initialize_memory(B, device)

    C = 67; A = 65; T = 84; R = 82; B_byte = 66

    print("="*50)
    print("FINAL ARCHITECTURE DIAGNOSTIC (Sum Aggregator)")
    print("="*50)

    with torch.no_grad():
        print("\n--- Processing CAT ---")
        _, motif_cat, mem_cat = model(cat_seq, fresh_memory.clone(), disable_gating=False, verbose=True)
        print("\n--- Processing CAR ---")
        car_seq = torch.tensor([[C, A, R]], dtype=torch.long).to(device)
        _, motif_car, mem_car = model(car_seq, fresh_memory.clone(), disable_gating=False, verbose=True)
        print("\n--- Processing BAT ---")
        bat_seq = torch.tensor([[B_byte, A, T]], dtype=torch.long).to(device)
        _, motif_bat, mem_bat = model(bat_seq, fresh_memory.clone(), disable_gating=False, verbose=True)

    def cos_sim(v1, v2):
        return F.cosine_similarity(v1.flatten().unsqueeze(0), v2.flatten().unsqueeze(0)).item()

    print("\n--- 1. NEURON-LEVEL SEPARATION (Internal Node 'A') ---")
    h_A_cat = mem_cat[0, A]
    h_A_car = mem_car[0, A]
    h_A_bat = mem_bat[0, A]
    print(f"sim(h_A in CAT, h_A in CAR): {cos_sim(h_A_cat, h_A_car):.4f}")
    print(f"sim(h_A in CAT, h_A in BAT): {cos_sim(h_A_cat, h_A_bat):.4f}")

    print("\n--- 2. MOTIF-LEVEL SEPARATION (Aggregator Output) ---")
    print(f"sim(Motif_CAT, Motif_CAR): {cos_sim(motif_cat, motif_car):.4f}")
    print(f"sim(Motif_CAT, Motif_BAT): {cos_sim(motif_cat, motif_bat):.4f}")
    print(f"sim(Motif_CAR, Motif_BAT): {cos_sim(motif_car, motif_bat):.4f}")

if __name__ == "__main__":
    run_final_diagnostics()

Training on 'CAT' (50 steps) with Sum Aggregator...
Training Complete.

FINAL ARCHITECTURE DIAGNOSTIC (Sum Aggregator)

--- Processing CAT ---
    [Aggregator Input Norms]: ['28.68', '14.58', '14.35', '14.15']

--- Processing CAR ---
    [Aggregator Input Norms]: ['28.68', '14.58', '170.39', '14.16']

--- Processing BAT ---
    [Aggregator Input Norms]: ['28.54', '15.18', '15.21', '14.15']

--- 1. NEURON-LEVEL SEPARATION (Internal Node 'A') ---
sim(h_A in CAT, h_A in CAR): 0.5924
sim(h_A in CAT, h_A in BAT): 0.5101

--- 2. MOTIF-LEVEL SEPARATION (Aggregator Output) ---
sim(Motif_CAT, Motif_CAR): 0.4591
sim(Motif_CAT, Motif_BAT): 0.6903
sim(Motif_CAR, Motif_BAT): 0.2788


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass
import math

@dataclass
class ModelConfig:
    d_model: int = 64
    vocab_size: int = 256
    max_archive_size: int = 500
    vigilance_threshold: float = 0.85
    lr: float = 1e-3
    alpha: float = 1.0

class DualSlotNeurons(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.d_model = config.d_model
        self.byte_emb = nn.Embedding(config.vocab_size + 2, self.d_model)
        self.W_before = nn.Linear(self.d_model, self.d_model)
        self.W_after = nn.Linear(self.d_model, self.d_model)
        self.norm_before = nn.LayerNorm(self.d_model)
        self.norm_after = nn.LayerNorm(self.d_model)
        self.norm_flush = nn.LayerNorm(self.d_model * 3)

    def initialize_memory(self, B, device):
        memory = torch.zeros(B, 256, 2, self.d_model, device=device)
        sos_emb = self.byte_emb(torch.tensor([256], device=device))
        eos_emb = self.byte_emb(torch.tensor([257], device=device))
        for i in range(256):
            self_emb = self.byte_emb(torch.tensor([i], device=device))
            memory[0, i, 0] = self.norm_before(self.W_before(sos_emb))
            memory[0, i, 1] = self.norm_after(self.W_after(eos_emb))
        return memory

    def forward(self, x_curr, prev_byte, memory_neurons):
        B = x_curr.shape[0]
        device = x_curr.device
        batch_idx = torch.arange(B, device=device)

        new_memory = memory_neurons.clone()
        curr_emb = self.byte_emb(x_curr)

        if prev_byte is not None:
            prev_emb = self.byte_emb(prev_byte)
            new_memory[batch_idx, prev_byte, 1] = self.norm_after(self.W_after(curr_emb))
            new_memory[batch_idx, x_curr, 0] = self.norm_before(self.W_before(prev_emb))

            h_before_prev = new_memory[batch_idx, prev_byte, 0]
            h_after_prev = new_memory[batch_idx, prev_byte, 1]

            raw_flush = torch.cat([h_before_prev, prev_emb, h_after_prev], dim=-1)
            normed_flush = self.norm_flush(raw_flush)

        else:
            sos_emb = self.byte_emb(torch.tensor([256], device=device))
            eos_emb = self.byte_emb(torch.tensor([257], device=device))
            new_memory[batch_idx, x_curr, 0] = self.norm_before(self.W_before(sos_emb))
            raw_flush = torch.cat([self.W_before(sos_emb), curr_emb, self.W_after(eos_emb)], dim=-1)
            normed_flush = self.norm_flush(raw_flush)

        return normed_flush, new_memory

class SumAggregator(nn.Module):
    def __init__(self, config):
        super().__init__()

    def forward(self, gated_flushes):
        motif_vector = torch.sum(gated_flushes, dim=1)
        return motif_vector

class EDSSM_Final(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.neurons = DualSlotNeurons(config)
        self.aggregator = SumAggregator(config)
        self.lm_head = nn.Linear(config.d_model * 3, config.vocab_size)

    def forward(self, x_seq, memory_neurons, disable_gating=False):
        B, T = x_seq.shape
        device = x_seq.device

        prev_byte = None
        prev_logits = None
        gated_flushes = []
        logits_list = []

        for t in range(T):
            x_t = x_seq[:, t]
            normed_flush, memory_neurons = self.neurons(x_t, prev_byte, memory_neurons)

            if prev_logits is not None:
                prob = F.softmax(prev_logits, dim=-1)[0, x_t.item()].item()
                surprise = -math.log(prob + 1e-8)
            else:
                surprise = 1.0

            gate = 1.0 if disable_gating else 1.0 + (self.config.alpha * surprise)
            gated_flush = normed_flush * gate
            gated_flushes.append(gated_flush)

            curr_logits = self.lm_head(normed_flush)
            logits_list.append(curr_logits)

            prev_logits = curr_logits
            prev_byte = x_t

        eos_emb = self.neurons.byte_emb(torch.tensor([257], device=device))
        last_emb = self.neurons.byte_emb(prev_byte)

        h_last_before = memory_neurons[torch.arange(B, device=device), prev_byte, 0]
        h_last_after = self.neurons.W_after(eos_emb)

        flush_input_eos = torch.cat([h_last_before, last_emb, h_last_after], dim=-1)
        normed_flush_eos = self.neurons.norm_flush(flush_input_eos)
        gated_flushes.append(normed_flush_eos * 1.0)

        flush_tensor = torch.stack(gated_flushes, dim=1)
        motif_vector = self.aggregator(flush_tensor)

        logits = torch.stack(logits_list, dim=1)
        return logits, motif_vector, memory_neurons

# ==========================================
# TEST 1: 4-PATH TOPOLOGY MATRIX
# ==========================================
def run_topology_test():
    config = ModelConfig()
    model = EDSSM_Final(config)
    optimizer = torch.optim.Adam(model.parameters(), lr=config.lr)
    criterion = nn.CrossEntropyLoss()

    B = 1
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.to(device)

    memory_neurons = model.neurons.initialize_memory(B, device)

    C = 67; A = 65; T = 84; R = 82; B_byte = 66

    cat_seq = torch.tensor([[C, A, T]], dtype=torch.long).to(device)
    car_seq = torch.tensor([[C, A, R]], dtype=torch.long).to(device)
    bat_seq = torch.tensor([[B_byte, A, T]], dtype=torch.long).to(device)
    bar_seq = torch.tensor([[B_byte, A, R]], dtype=torch.long).to(device)

    sequences = [cat_seq, car_seq, bat_seq, bar_seq]
    names = ["CAT", "CAR", "BAT", "BAR"]

    print("Training on CAT, CAR, BAT, BAR (50 steps each)...")

    for step in range(50):
        for seq in sequences:
            optimizer.zero_grad()
            logits, _, memory_neurons = model(seq, memory_neurons, disable_gating=False)
            memory_neurons = memory_neurons.detach()

            targets = seq[:, 1:]
            # FIX: Removed config.vocab_size from targets view
            loss = criterion(logits[:, :-1].contiguous().view(-1, config.vocab_size), targets.contiguous().view(-1))
            loss.backward()
            optimizer.step()

    print("Training Complete.\n")

    print("="*50)
    print("TEST 1: MOTIF TOPOLOGY MATRIX")
    print("="*50)

    motifs = {}
    with torch.no_grad():
        fresh_memory = model.neurons.initialize_memory(B, device)
        for name, seq in zip(names, sequences):
            _, motif, _ = model(seq, fresh_memory.clone(), disable_gating=False)
            motifs[name] = motif

    def cos_sim(v1, v2):
        return F.cosine_similarity(v1.flatten().unsqueeze(0), v2.flatten().unsqueeze(0)).item()

    print("\nMotif Similarity Matrix:")
    print("      ", "  ".join([f"{n:>6}" for n in names]))
    for n1 in names:
        row = [f"{n1:>6}"]
        for n2 in names:
            sim = cos_sim(motifs[n1], motifs[n2])
            row.append(f"{sim:6.4f}")
        print("  ".join(row))

    print("\nStructural Analysis:")
    sim_cat_car = cos_sim(motifs["CAT"], motifs["CAR"])
    sim_bat_bar = cos_sim(motifs["BAT"], motifs["BAR"])
    sim_cat_bat = cos_sim(motifs["CAT"], motifs["BAT"])
    sim_car_bar = cos_sim(motifs["CAR"], motifs["BAR"])
    sim_cat_bar = cos_sim(motifs["CAT"], motifs["BAR"])

    print(f"Shared 2 edges (CAT↔CAR, BAT↔BAR): {(sim_cat_car+sim_bat_bar)/2:.4f} (Expected: Highest)")
    print(f"Shared 1 edge (CAT↔BAT, CAR↔BAR): {(sim_cat_bat+sim_car_bar)/2:.4f} (Expected: Medium)")
    print(f"Shared 0 edges (CAT↔BAR): {sim_cat_bar:.4f} (Expected: Lowest)")

if __name__ == "__main__":
    run_topology_test()

Training on CAT, CAR, BAT, BAR (50 steps each)...
Training Complete.

TEST 1: MOTIF TOPOLOGY MATRIX

Motif Similarity Matrix:
          CAT     CAR     BAT     BAR
   CAT  1.0000  0.8923  0.6998  0.5654
   CAR  0.8923  1.0000  0.5825  0.6491
   BAT  0.6998  0.5825  1.0000  0.9043
   BAR  0.5654  0.6491  0.9043  1.0000

Structural Analysis:
Shared 2 edges (CAT↔CAR, BAT↔BAR): 0.8983 (Expected: Highest)
Shared 1 edge (CAT↔BAT, CAR↔BAR): 0.6745 (Expected: Medium)
Shared 0 edges (CAT↔BAR): 0.5654 (Expected: Lowest)


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass
import math

@dataclass
class ModelConfig:
    d_model: int = 64
    vocab_size: int = 256
    max_archive_size: int = 500
    vigilance_threshold: float = 0.85
    lr: float = 1e-3
    alpha: float = 1.0

class DualSlotNeurons(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.d_model = config.d_model
        self.byte_emb = nn.Embedding(config.vocab_size + 2, self.d_model)
        self.W_before = nn.Linear(self.d_model, self.d_model)
        self.W_after = nn.Linear(self.d_model, self.d_model)
        self.norm_before = nn.LayerNorm(self.d_model)
        self.norm_after = nn.LayerNorm(self.d_model)
        self.norm_flush = nn.LayerNorm(self.d_model * 3)

    def initialize_memory(self, B, device):
        memory = torch.zeros(B, 256, 2, self.d_model, device=device)
        sos_emb = self.byte_emb(torch.tensor([256], device=device))
        eos_emb = self.byte_emb(torch.tensor([257], device=device))
        for i in range(256):
            self_emb = self.byte_emb(torch.tensor([i], device=device))
            memory[0, i, 0] = self.norm_before(self.W_before(sos_emb))
            memory[0, i, 1] = self.norm_after(self.W_after(eos_emb))
        return memory

    def forward(self, x_curr, prev_byte, memory_neurons):
        B = x_curr.shape[0]
        device = x_curr.device
        batch_idx = torch.arange(B, device=device)

        new_memory = memory_neurons.clone()
        curr_emb = self.byte_emb(x_curr)

        if prev_byte is not None:
            prev_emb = self.byte_emb(prev_byte)
            new_memory[batch_idx, prev_byte, 1] = self.norm_after(self.W_after(curr_emb))
            new_memory[batch_idx, x_curr, 0] = self.norm_before(self.W_before(prev_emb))

            h_before_prev = new_memory[batch_idx, prev_byte, 0]
            h_after_prev = new_memory[batch_idx, prev_byte, 1]

            raw_flush = torch.cat([h_before_prev, prev_emb, h_after_prev], dim=-1)
            normed_flush = self.norm_flush(raw_flush)

        else:
            sos_emb = self.byte_emb(torch.tensor([256], device=device))
            eos_emb = self.byte_emb(torch.tensor([257], device=device))
            new_memory[batch_idx, x_curr, 0] = self.norm_before(self.W_before(sos_emb))
            raw_flush = torch.cat([self.W_before(sos_emb), curr_emb, self.W_after(eos_emb)], dim=-1)
            normed_flush = self.norm_flush(raw_flush)

        return normed_flush, new_memory

class SumAggregator(nn.Module):
    def __init__(self, config):
        super().__init__()

    def forward(self, gated_flushes):
        motif_vector = torch.sum(gated_flushes, dim=1)
        return motif_vector

class EDSSM_Final(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.neurons = DualSlotNeurons(config)
        self.aggregator = SumAggregator(config)
        self.lm_head = nn.Linear(config.d_model * 3, config.vocab_size)

    def forward(self, x_seq, memory_neurons, disable_gating=False):
        B, T = x_seq.shape
        device = x_seq.device

        prev_byte = None
        prev_logits = None
        gated_flushes = []
        logits_list = []

        for t in range(T):
            x_t = x_seq[:, t]
            normed_flush, memory_neurons = self.neurons(x_t, prev_byte, memory_neurons)

            if prev_logits is not None:
                prob = F.softmax(prev_logits, dim=-1)[0, x_t.item()].item()
                surprise = -math.log(prob + 1e-8)
            else:
                surprise = 1.0

            gate = 1.0 if disable_gating else 1.0 + (self.config.alpha * surprise)
            gated_flush = normed_flush * gate
            gated_flushes.append(gated_flush)

            curr_logits = self.lm_head(normed_flush)
            logits_list.append(curr_logits)

            prev_logits = curr_logits
            prev_byte = x_t

        eos_emb = self.neurons.byte_emb(torch.tensor([257], device=device))
        last_emb = self.neurons.byte_emb(prev_byte)

        h_last_before = memory_neurons[torch.arange(B, device=device), prev_byte, 0]
        h_last_after = self.neurons.W_after(eos_emb)

        flush_input_eos = torch.cat([h_last_before, last_emb, h_last_after], dim=-1)
        normed_flush_eos = self.neurons.norm_flush(flush_input_eos)
        gated_flushes.append(normed_flush_eos * 1.0)

        flush_tensor = torch.stack(gated_flushes, dim=1)
        motif_vector = self.aggregator(flush_tensor)

        logits = torch.stack(logits_list, dim=1)
        return logits, motif_vector, memory_neurons

# ==========================================
# TEST 4: DISJOINT NODE PERMUTATION TEST
# ==========================================
def run_permutation_test():
    config = ModelConfig()
    model = EDSSM_Final(config)
    optimizer = torch.optim.Adam(model.parameters(), lr=config.lr)
    criterion = nn.CrossEntropyLoss()

    B = 1
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.to(device)

    memory_neurons = model.neurons.initialize_memory(B, device)

    C = 67; A = 65; T = 84
    W = 87; P = 80; S = 83

    cat_seq = torch.tensor([[C, A, T]], dtype=torch.long).to(device)
    wps_seq = torch.tensor([[W, P, S]], dtype=torch.long).to(device)

    sequences = [cat_seq, wps_seq]
    names = ["CAT", "WPS"]

    print("Training on CAT, WPS (50 steps each)...")

    for step in range(50):
        for seq in sequences:
            optimizer.zero_grad()
            logits, _, memory_neurons = model(seq, memory_neurons, disable_gating=False)
            memory_neurons = memory_neurons.detach()

            targets = seq[:, 1:]
            loss = criterion(logits[:, :-1].contiguous().view(-1, config.vocab_size), targets.contiguous().view(-1))
            loss.backward()
            optimizer.step()

    print("Training Complete.\n")

    print("="*50)
    print("TEST 4: DISJOINT NODE PERMUTATION TEST")
    print("="*50)

    motifs = {}
    with torch.no_grad():
        fresh_memory = model.neurons.initialize_memory(B, device)
        for name, seq in zip(names, sequences):
            _, motif, _ = model(seq, fresh_memory.clone(), disable_gating=False)
            motifs[name] = motif

    def cos_sim(v1, v2):
        return F.cosine_similarity(v1.flatten().unsqueeze(0), v2.flatten().unsqueeze(0)).item()

    sim_cat_wps = cos_sim(motifs["CAT"], motifs["WPS"])
    print(f"\nsim(Motif_CAT, Motif_WPS): {sim_cat_wps:.4f}")
    print("\nAnalysis:")
    if sim_cat_wps < 0.4:
        print("PASS: Architecture achieves strong separation for disjoint paths.")
        print("The 0.57 seen earlier was likely due to shared internal node 'A'.")
    elif sim_cat_wps < 0.6:
        print("PARTIAL: Some separation, but a high baseline correlation remains.")
        print("Likely caused by shared SOS/EOS tokens or LayerNorm centering.")
    else:
        print("FAIL: Architecture cannot fully separate disjoint paths.")

if __name__ == "__main__":
    run_permutation_test()

Training on CAT, WPS (50 steps each)...
Training Complete.

TEST 4: DISJOINT NODE PERMUTATION TEST

sim(Motif_CAT, Motif_WPS): 0.3776

Analysis:
PASS: Architecture achieves strong separation for disjoint paths.
The 0.57 seen earlier was likely due to shared internal node 'A'.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass
import math

@dataclass
class ModelConfig:
    d_model: int = 64
    vocab_size: int = 256
    max_archive_size: int = 500
    vigilance_threshold: float = 0.85
    lr: float = 1e-3
    alpha: float = 1.0

class DualSlotNeurons(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.d_model = config.d_model
        self.byte_emb =